In [ ]:
-- 실행 컨텍스트 설정
USE ROLE ACCOUNTADMIN;
USE DATABASE DEMO;
USE SCHEMA ML_DEMO;


# Module 7: ML Observability

프로덕션 모델의 성능 저하를 감지하고 재학습 트리거를 구현합니다.

### 학습 목표
- Model Monitor 설정
- 피처 드리프트(PSI) 시뮬레이션 및 감지
- 재학습 트리거 로직


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
import snowflake.snowpark.types as T
from snowflake.ml.registry import Registry

session = get_active_session()
session.use_database('DEMO')
session.use_schema('ML_DEMO')
print('Session ready')


## 1. Inference Log 생성 (시뮬레이션)

모델 추론 결과를 로깅하는 테이블을 생성합니다. 드리프트 시뮬레이션을 위해 일부 데이터에 노이즈를 추가합니다.


In [ ]:
# Inference Log 테이블 생성 (30일 시뮬레이션)
reg = Registry(session=session, database_name='DEMO', schema_name='ML_DEMO')
mv = reg.get_model('CUSTOMER_LTV_PREDICTOR').version('V1')

# Managed FV에서 최신 피처 조회
fv_df = session.table('DEMO.ML_DEMO.CUSTOMER_LTV_FEATURES$1')
latest_date = fv_df.select(F.max('FEATURE_DATE')).collect()[0][0]
base_df = fv_df.filter(F.col('FEATURE_DATE') == latest_date).limit(1000)

# 추론 실행
preds = mv.run(base_df, function_name='predict')
pred_pd = preds.select('C_CUSTKEY', 'PREDICTED_LABEL', 'TOTAL_ORDERS', 'AVG_ORDER_VALUE').to_pandas()

# 30일 시뮬레이션 (후반 10일에 드리프트 추가)
np.random.seed(42)
records = []
for day in range(30):
    day_df = pred_pd.copy()
    date = pd.Timestamp('1997-06-01') + pd.Timedelta(days=day)
    day_df['PREDICTED_LTV_TIME'] = date
    day_df['ACTUAL_LABEL'] = day_df['PREDICTED_LABEL'] * np.random.uniform(0.8, 1.2, len(day_df))
    # 드리프트: 후반 10일에 피처 분포 변경
    if day >= 20:
        day_df['TOTAL_ORDERS'] = day_df['TOTAL_ORDERS'] * np.random.uniform(1.5, 2.5, len(day_df))
        day_df['AVG_ORDER_VALUE'] = day_df['AVG_ORDER_VALUE'] * np.random.uniform(0.3, 0.7, len(day_df))
    records.append(day_df)

log_pd = pd.concat(records, ignore_index=True)
log_df = session.create_dataframe(log_pd)
log_df.write.mode('overwrite').save_as_table('DEMO.ML_DEMO.INFERENCE_LOG')
print(f'INFERENCE_LOG: {log_df.count():,} rows (30 days x 1,000 customers)')


## 2. 성능 모니터링


In [ ]:
# 일별 성능 메트릭 계산
log_df = session.table('DEMO.ML_DEMO.INFERENCE_LOG')
metrics_pd = log_df.group_by(F.col('PREDICTED_LTV_TIME').cast(T.DateType()).alias('DATE')).agg(
    F.avg(F.abs(F.col('PREDICTED_LABEL') - F.col('ACTUAL_LABEL'))).alias('MAE'),
    F.count('*').alias('COUNT')
).order_by('DATE').to_pandas()

# 시각화
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(metrics_pd['DATE'], metrics_pd['MAE'], 'o-', color='#636EFA')
ax.axvline(x=metrics_pd['DATE'].iloc[20], color='red', linestyle='--', label='Drift Start')
ax.set_title('Daily MAE Trend')
ax.set_xlabel('Date')
ax.set_ylabel('MAE ($)')
ax.legend()
plt.tight_layout()
plt.show()


## 3. 피처 드리프트 감지 (PSI)


In [ ]:
# PSI (Population Stability Index) 계산
def calc_psi(baseline, current, bins=10):
    hist_b, edges = np.histogram(baseline, bins=bins, density=True)
    hist_c, _ = np.histogram(current, bins=edges, density=True)
    hist_b = np.clip(hist_b, 1e-6, None)
    hist_c = np.clip(hist_c, 1e-6, None)
    return float(np.sum((hist_c - hist_b) * np.log(hist_c / hist_b)))

log_pd = session.table('DEMO.ML_DEMO.INFERENCE_LOG').to_pandas()
baseline = log_pd[log_pd['PREDICTED_LTV_TIME'] < '1997-06-21']
current = log_pd[log_pd['PREDICTED_LTV_TIME'] >= '1997-06-21']

features = ['TOTAL_ORDERS', 'AVG_ORDER_VALUE']
psi_results = []
for feat in features:
    psi = calc_psi(baseline[feat].values, current[feat].values)
    status = 'HIGH DRIFT' if psi >= 0.20 else 'WARNING' if psi >= 0.10 else 'STABLE'
    psi_results.append({'Feature': feat, 'PSI': psi, 'Status': status})

drift_df = pd.DataFrame(psi_results)
print(drift_df.to_string(index=False))


In [ ]:
# PSI 시각화
fig, ax = plt.subplots(figsize=(8, 3))
colors = ['#f85149' if s == 'HIGH DRIFT' else '#ffa657' if s == 'WARNING' else '#3fb950'
          for s in drift_df['Status']]
ax.barh(drift_df['Feature'], drift_df['PSI'], color=colors)
ax.axvline(x=0.20, color='red', linestyle='--', label='Danger (0.20)')
ax.axvline(x=0.10, color='orange', linestyle='--', label='Warning (0.10)')
ax.set_title('Feature Drift (PSI)')
ax.set_xlabel('PSI')
ax.legend()
plt.tight_layout()
plt.show()


## 4. 재학습 트리거


In [ ]:
# 재학습 필요 여부 판단
n_high_drift = len(drift_df[drift_df['Status'] == 'HIGH DRIFT'])
max_psi = drift_df['PSI'].max()
latest_mae = metrics_pd['MAE'].iloc[-1]

retrain_needed = (n_high_drift >= 1 or max_psi > 0.30 or latest_mae > 50000)

print(f'HIGH DRIFT features: {n_high_drift}')
print(f'Max PSI: {max_psi:.4f}')
print(f'Latest MAE: {latest_mae:,.0f}')
print(f'\nRetrain needed: {retrain_needed}')
print(f'Action: {"Execute retrain" if retrain_needed else "Continue monitoring"}')


## 정리

| PSI 범위 | 상태 | 조치 |
|----------|------|------|
| < 0.10 | STABLE | 모니터링 유지 |
| 0.10~0.20 | WARNING | 원인 분석 |
| >= 0.20 | HIGH DRIFT | 재학습 권고 |
